In [32]:
import torch
import torch.nn as nn

In [33]:
class BERTEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=512):
        super().__init__()
        # 三种embedding：词、段落、位置
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.segment_emb = nn.Embedding(2, d_model)
        self.position_emb = nn.Embedding(max_len, d_model)

        # BERT在Embedding后习惯加一个LayerNorm和Dropout
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, segment_ids):
        seq_len = input_ids.shape[1]

        position_ids = torch.arange(seq_len, dtype=torch.long)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)

        embeddings = self.token_emb(input_ids) + self.segment_emb(segment_ids) + self.position_emb(position_ids)


        return self.dropout(self.norm(embeddings))

In [34]:
class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.gelu = nn.GELU()
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.gelu(self.linear1(x)))

In [35]:
import math
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads=8):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)

        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        # q.shape:[batch_size, seq, d_model]
        batch_size = q.shape[0]
        Q = self.W_Q(q)
        K = self.W_K(k)
        V = self.W_V(v)

        Q = Q.view(batch_size, -1, self.num_heads, self.d_k)
        K = K.view(batch_size, -1, self.num_heads, self.d_k)
        V = V.view(batch_size, -1, self.num_heads, self.d_k)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        K_transposed = K.transpose(-2, -1)

        scores = torch.matmul(Q, K_transposed) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)

        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, -1, self.d_model)

        return self.W_O(context)

In [38]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model=512, d_ff=2048, num_heads=8):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.layer_norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        attn_weight = self.mha(x, x, x, mask)
        x = self.layer_norm1(x + attn_weight)
        ffn_output = self.ffn(x)
        x = self.layer_norm2(x + ffn_output)

        return x

In [39]:
class BERT(nn.Module):
    def __init__(self, vocab_size, d_model=512, d_ff=2048, num_heads=8, num_layers=12):
        super().__init__()
        self.embedding = BERTEmbedding(vocab_size, d_model)
        self.encoder = nn.ModuleList([
            EncoderBlock(d_model=d_model, d_ff=d_ff, num_heads=num_heads) for _ in range(num_layers)
        ])

        self.mlm_head = nn.Linear(d_model, vocab_size)
        self.nsp_head = nn.Linear(d_model, 2)

    def forward(self, input_ids, segment_ids):
        x = self.embedding(input_ids, segment_ids)

        for layer in self.encoder:
            x = layer(x)

        mlm_logits = self.mlm_head(x)

        cls_vector = x[:, 0, :]
        nsp_logits = self.nsp_head(cls_vector)

        return mlm_logits, nsp_logits

In [40]:
import torch

# 假设普通词的 ID 随便定几个
# 我=10, 爱=11, 学=12, 特=20, 别=21, 开=22, 心=23
# [PAD]=0, [CLS]=1, [SEP]=2, [MASK]=3

# 1. 构造 input_ids (词的身份)
# 序列: [CLS, 我, 爱, 学, MASK, SEP, 特, 别, 开, 心, SEP, PAD]
input_sequence = [1, 10, 11, 12, 3, 2, 20, 21, 22, 23, 2, 0]

# 我们把它变成一个 batch_size=1 的二维张量: [1, 12]
input_ids = torch.tensor([input_sequence]) 

# 2. 构造 segment_ids (告诉模型谁是第一句，谁是第二句)
# 规则: 第一个 [SEP] 及之前全是 0，之后到第二个 [SEP] 全是 1，PAD 设为 0
segment_sequence = [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0]

# 同样变成张量: [1, 12]
segment_ids = torch.tensor([segment_sequence])

print("1. 模型看到的词汇 ID:\n", input_ids)
print("\n2. 模型看到的句子归属 (0=句A, 1=句B):\n", segment_ids)

1. 模型看到的词汇 ID:
 tensor([[ 1, 10, 11, 12,  3,  2, 20, 21, 22, 23,  2,  0]])

2. 模型看到的句子归属 (0=句A, 1=句B):
 tensor([[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0]])


In [41]:
# 假装实例化了模型
# vocab_size=100 (字典大小)
bert_model = BERT(vocab_size=100, d_model=64, num_heads=2, num_layers=2)

# 关闭梯度，进入推理模式
bert_model.eval()

with torch.no_grad():
    # 输入数据穿过 2 层 Encoder
    mlm_logits, nsp_logits = bert_model(input_ids, segment_ids)

print("\n--- BERT 预测结果 ---")

# --- 分析 1: NSP (下一句预测) ---
print("3. NSP 头输出的形状:", nsp_logits.shape) # 预期: [1, 2]
print("   NSP 原始得分 (Logits):", nsp_logits[0])
# 思考：这个 [1, 2] 就是二分类得分。比如 [4.5, -1.2] 代表模型极度确信 Sentence B 
# 就是 Sentence A 的下一句 (因为索引 0 代表 True 的概率高)。

# --- 分析 2: MLM (掩码完形填空) ---
print("\n4. MLM 头输出的形状:", mlm_logits.shape) # 预期: [1, 12, 100]
# 思考：这是一个极其庞大的张量！对于这 12 个位置的每一个词，模型都在字典的 100 个词里打了个分。

# 我们把目光死死锁定在 [MASK] 这个词的位置！
# 在 input_sequence 中，[MASK] 是第 4 个元素 (索引为 4)
mask_position = 4
mask_word_logits = mlm_logits[0, mask_position, :] # 提取这个位置的 100 个打分

# 找出这 100 个分数里最高的那一个的索引
predicted_word_id = mask_word_logits.argmax(dim=-1).item()

print(f"5. [MASK] 所在位置的预测词汇 ID 是: {predicted_word_id}")
# 思考：如果这是一个被海量数据训练过的成熟 BERT，这个 predicted_word_id 
# 大概率会等于我们心中想的那个“习”字的 ID！


--- BERT 预测结果 ---
3. NSP 头输出的形状: torch.Size([1, 2])
   NSP 原始得分 (Logits): tensor([-0.0562, -0.4106])

4. MLM 头输出的形状: torch.Size([1, 12, 100])
5. [MASK] 所在位置的预测词汇 ID 是: 95


In [42]:
import torch
import random
from torch.utils.data import Dataset

# --- 1. 模拟一个微型字典和特殊标记 ---
PAD_ID = 0
CLS_ID = 1
SEP_ID = 2
MASK_ID = 3
VOCAB_SIZE = 1000 # 假设字典里有 1000 个词

class BERTPretrainDataset(Dataset):
    def __init__(self, corpus, max_len=30):
        """
        corpus: 一个列表，里面装着所有的句子。为了演示，假设句子已经是 ID 列表了。
                例如: [[10, 22, 45], [88, 12, 9], ...]
        max_len: 模型能接收的最大长度
        """
        self.corpus = corpus
        self.max_len = max_len
        self.corpus_size = len(corpus)

    def __len__(self):
        return self.corpus_size

    def __getitem__(self, idx):
        # ==========================================
        # 动作一：NSP (下一句预测) 数据构造
        # ==========================================
        sentence_a = self.corpus[idx]
        
        # 50% 的概率：老老实实拿真实的下一句
        if random.random() > 0.5 and idx + 1 < self.corpus_size:
            sentence_b = self.corpus[idx + 1]
            nsp_label = 1 # True: 是连续的
        # 50% 的概率：去语料库里随便抓一句八竿子打不着的话
        else:
            random_idx = random.randint(0, self.corpus_size - 1)
            sentence_b = self.corpus[random_idx]
            nsp_label = 0 # False: 不是连续的

        # ==========================================
        # 动作二：拼接成 BERT 要求的专属格式
        # ==========================================
        # 格式: [CLS] Sentence A [SEP] Sentence B [SEP]
        tokens = [CLS_ID] + sentence_a + [SEP_ID] + sentence_b + [SEP_ID]
        
        # 对应的 segment_ids (句 A 部分是 0，句 B 部分是 1)
        # 注意：[CLS] 和第一个 [SEP] 都属于句 A 的势力范围
        segment_ids = [0] * (len(sentence_a) + 2) + [1] * (len(sentence_b) + 1)

        # ==========================================
        # 动作三：MLM (动态掩码) 核心逻辑！
        # ==========================================
        input_ids = []
        mlm_labels = [] # 用来装标准答案，如果是普通词就存 PAD_ID(忽略)，如果是被蒙住的词就存真实 ID

        for token in tokens:
            # 遇到特殊标记，绝对不能遮挡！直接放行
            if token in [CLS_ID, SEP_ID, PAD_ID]:
                input_ids.append(token)
                mlm_labels.append(PAD_ID) # 填 PAD_ID 代表算 Loss 的时候忽略这个位置
                continue
            
            # 对普通词抛硬币：15% 的概率被选中去“做手术”
            if random.random() < 0.15:
                mlm_labels.append(token) # 既然被选中了，就要把真实身份存入“答案小本本”
                
                prob = random.random()
                if prob < 0.8:
                    # 80% 的情况：乖乖变成 [MASK]
                    input_ids.append(MASK_ID)
                elif prob < 0.9:
                    # 10% 的情况：随机变成字典里的另外一个词 (制造噪音)
                    random_token = random.randint(4, VOCAB_SIZE - 1)
                    input_ids.append(random_token)
                else:
                    # 10% 的情况：保持原样，什么都不做，但依然要算 Loss！
                    input_ids.append(token)
            else:
                # 85% 的概率：安全过关，没有被选中
                input_ids.append(token)
                mlm_labels.append(PAD_ID) # 算 Loss 时忽略

        # ==========================================
        # 动作四：截断与填充 (Padding)
        # ==========================================
        # 如果太长，就咔嚓一刀切掉
        if len(input_ids) > self.max_len:
            input_ids = input_ids[:self.max_len]
            segment_ids = segment_ids[:self.max_len]
            mlm_labels = mlm_labels[:self.max_len]
        
        # 如果太短，就用 PAD_ID 补齐到 max_len
        padding_length = self.max_len - len(input_ids)
        if padding_length > 0:
            input_ids.extend([PAD_ID] * padding_length)
            segment_ids.extend([PAD_ID] * padding_length)
            mlm_labels.extend([PAD_ID] * padding_length)

        # 最终：将 Python 列表转化为 PyTorch 张量返回
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "segment_ids": torch.tensor(segment_ids, dtype=torch.long),
            "mlm_labels": torch.tensor(mlm_labels, dtype=torch.long),
            "nsp_label": torch.tensor(nsp_label, dtype=torch.long)
        }

In [43]:
# 模拟 3 句话，每句话用一串 ID 表示
mock_corpus = [
    [10, 11], # 句子 0
    [12, 13, 14],     # 句子 1
    [15, 16],
    [17, 18, 19, 20],  # 句子 2
    [21, 22, 23]
]

# 实例化数据集 (设定最大长度为 15)
dataset = BERTPretrainDataset(mock_corpus, max_len=15)
print(len(dataset))

# 我们强行从 DataLoader 的视角，拿第 0 条数据出来看看
for i in range(len(dataset)):
    sample = dataset[i]
    print("1. 送入模型的 input_ids:\n", sample["input_ids"])
    print("2. 区分句子的 segment_ids:\n", sample["segment_ids"])
    print("3. 完形填空的正确答案 mlm_labels (0代表不计算Loss):\n", sample["mlm_labels"])
    print("4. 两句话是否连续的 nsp_label:\n", sample["nsp_label"])
    print("-------------------------------")

5
1. 送入模型的 input_ids:
 tensor([ 1, 10, 11,  2, 10, 11,  2,  0,  0,  0,  0,  0,  0,  0,  0])
2. 区分句子的 segment_ids:
 tensor([0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0])
3. 完形填空的正确答案 mlm_labels (0代表不计算Loss):
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
4. 两句话是否连续的 nsp_label:
 tensor(0)
-------------------------------
1. 送入模型的 input_ids:
 tensor([ 1, 12,  3, 14,  2, 15, 16,  2,  0,  0,  0,  0,  0,  0,  0])
2. 区分句子的 segment_ids:
 tensor([0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0])
3. 完形填空的正确答案 mlm_labels (0代表不计算Loss):
 tensor([ 0,  0, 13,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])
4. 两句话是否连续的 nsp_label:
 tensor(1)
-------------------------------
1. 送入模型的 input_ids:
 tensor([ 1, 15, 16,  2, 17, 18, 19, 20,  2,  0,  0,  0,  0,  0,  0])
2. 区分句子的 segment_ids:
 tensor([0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
3. 完形填空的正确答案 mlm_labels (0代表不计算Loss):
 tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
4. 两句话是否连续的 nsp_label:
 tensor(1)
-------------------------------
1. 送入